## Using the trained model to predict in our dataset

### First we import the packages and the dataset

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

import pandas as pd

# Start by importing the test data
test_path = Path('data/raw/dataset_test.csv')
df_test = pd.read_csv(test_path)

Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Now we perform the datacleaning

In [2]:
from src.data_cleaning import clean_data

df_test = clean_data(dataset= df_test, is_train = False)

df_test

,id,datetime,station_number,name,lat,lng,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos,is_holiday
0,2025-03-13 08:30:00_32000,2025-03-13 08:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,8,30,3,3,0,0.866025,-0.500000,False
1,2025-03-13 08:30:00_32001,2025-03-13 08:30:00,32001,Hoher Markt,48.210666,16.372983,8,30,3,3,0,0.866025,-0.500000,False
2,2025-03-13 08:30:00_32002,2025-03-13 08:30:00,32002,Oper,48.202683,16.369702,8,30,3,3,0,0.866025,-0.500000,False
3,2025-03-13 08:30:00_32003,2025-03-13 08:30:00,32003,Volksgarten,48.206516,16.360400,8,30,3,3,0,0.866025,-0.500000,False
4,2025-03-13 08:30:00_32004,2025-03-13 08:30:00,32004,Taborstraße U2,48.219522,16.382218,8,30,3,3,0,0.866025,-0.500000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537440,2025-04-29 23:30:00_32275,2025-04-29 23:30:00,32275,eLastenräder - Am langen Felde,48.250224,16.450650,23,30,1,4,0,-0.258819,0.965926,False
537441,2025-04-29 23:30:00_32277,2025-04-29 23:30:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,23,30,1,4,0,-0.258819,0.965926,False
537442,2025-04-29 23:30:00_32278,2025-04-29 23:30:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,23,30,1,4,0,-0.258819,0.965926,False
537443,2025-04-29 23:30:00_32280,2025-04-29 23:30:00,32280,ALF Mobility-Point,48.251355,16.452810,23,30,1,4,0,-0.258819,0.965926,False


In [3]:
# Save IDs and do preprocessing (as we did before)

test_ids = df_test["id"].copy()      # save the ids

DROP_COLS = ["id", "datetime", "name", "lat", "lng"]

df_test = df_test.drop(columns = DROP_COLS)


### Do the predictions

In [4]:
import joblib
import pandas as pd

# Load the trained model
model_path = Path('models/decision_tree_v1.joblib')
model = joblib.load(model_path)

# Make predictions
predictions = model.predict(df_test)

# (Optional) For classifiers, get probability estimates
# probabilities = model.predict_proba(df_test)

### Prepare the submission

In [5]:
submission = pd.DataFrame({"id": test_ids, "bikes": predictions})

# and now save the predictions
from pathlib import Path

out_path = Path('reports/predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)